# Perfis químicos e estimativas de metais — WaterPath v2

**Pergunta:** como variam as concentrações de metais estimadas pelo WaterPath entre perfis químicos medianos dos países disponíveis no GEMStat GFQA v3?

Este notebook inspeciona a base local, calcula um perfil químico mediano por país com pelo menos 30 amostras e consulta o artefato v2 já treinado para estimar oito metais. Cada perfil é um cenário sintético agregado, não uma amostra observada específica. A análise não é previsão futura, comparação causal entre países ou avaliação independente do modelo.


## Contexto e fontes

WaterPath analisa dados de qualidade da água e imagens dos corpos hídricos. Este notebook cobre apenas a regressão de metais: `MultiOutputRegressor`, transformação quantílica do alvo e, para cada metal, `StackingRegressor` com Random Forest, XGBoost e CatBoost combinados por RidgeCV. Veja o [relatório do experimento](../training/experiment_outputs/regression/metals_v2/README.md) e o [notebook de treinamento](../training/notebooks/regression/MultiOutputRegressor_v2.ipynb).

Os dados são do **UNEP GEMS/Water Global Freshwater Quality Archive (GFQA v3)**, publicado no Zenodo sob licença **CC BY 4.0**. A atribuição do manifesto é Heinle, Moritz; Lisniak, Dmytro; Saile, Philipp; UNEP GEMS/Water Data Centre. A base preparada contém 14.034 registros, 777 estações e nove países. A resolução espacial é por estação; a temporal é a data/hora da amostra. O notebook não usa o recorte local do Rio Doce, que não contém todas as entradas obrigatórias.


## Dados, método e ensemble

O modelo recebe temperatura (°C), pH, condutividade elétrica (µS/cm), oxigênio dissolvido (mg/L), sólidos suspensos totais (mg/L), carbono orgânico total (mg/L) e fósforo total (µg/L). As quatro primeiras entradas são obrigatórias. As saídas são Fe e Mn (mg/L), e Cr, Ni, Cu, Zn, Cd e Pb (µg/L).

O ensemble é demonstrado porque o relatório v2 registra erro absoluto médio normalizado de 0,130471, contra 0,138177 do Random Forest anterior (redução de 5,58%) e 0,172625 da mediana, no mesmo conjunto de teste. A mistura de modelos de árvores com padrões distintos e RidgeCV como combinador teve ganho nessa avaliação; isso não demonstra superioridade universal. A comparação reutiliza um teste já avaliado antes e é exploratória.

O notebook calcula perfis medianos por país (mínimo de 30 amostras), ignorando nulos. País não é entrada do modelo. Diferenças nos resultados podem refletir as entradas e a cobertura da base; não indicam efeito causal ou qualidade geral da água.


In [ ]:
from pathlib import Path
import hashlib
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root():
    for start in (Path.cwd().resolve(), Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "back-end/ia/app/models/metals_v2/quality_model.joblib").exists():
                return candidate
    raise FileNotFoundError("Execute a partir da raiz do repositório ou de back-end/ia/notes.")

REPO = find_repo_root()
NOTES = REPO / "back-end/ia/notes"
DATA_PATH = NOTES / "data/samples.csv"
DATA_MANIFEST_PATH = NOTES / "data/dataset_manifest.json"
MODEL_PATH = REPO / "back-end/ia/app/models/metals_v2/quality_model.joblib"
MODEL_MANIFEST_PATH = MODEL_PATH.with_name("manifest.json")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

data_manifest = json.loads(DATA_MANIFEST_PATH.read_text(encoding="utf-8"))
model_manifest = json.loads(MODEL_MANIFEST_PATH.read_text(encoding="utf-8"))
assert sha256(DATA_PATH) == data_manifest["samples_sha256"], "Hash do CSV diverge do manifesto"
assert sha256(MODEL_PATH) == model_manifest["model_sha256"], "Hash do modelo diverge do manifesto"
print("Dataset:", data_manifest["version"], "| licença:", data_manifest["license"])
print("Modelo:", model_manifest["version"])


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
display(df.dtypes.rename("tipo").to_frame())
display(df.isna().sum().rename("ausentes").to_frame().sort_values("ausentes", ascending=False))

for column in ["date", "station", "country", "sample_id"]:
    assert column in df.columns, f"Coluna ausente: {column}"
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("Período:", df["date"].min().date(), "a", df["date"].max().date())
print("Estações:", df["station"].nunique(), "| países:", df["country"].nunique())
print("IDs duplicados:", int(df["sample_id"].duplicated().sum()))
display(df["country"].value_counts().rename_axis("país").to_frame("amostras"))


## Cobertura e cenários medianos

As quatro entradas obrigatórias devem estar preenchidas para cada cenário. As pipelines do modelo imputam entradas opcionais ausentes com estatísticas aprendidas no treinamento. Países com menos de 30 amostras ficam de fora para reduzir a instabilidade de perfis baseados em pouquíssimas observações.


In [ ]:
model = joblib.load(MODEL_PATH)
feature_columns = list(model.feature_names_in_)
target_columns = model_manifest["targets"]
required_features = ["T, °C", "рН", "EC, µS/cm", "DO, mg/L"]
missing = sorted(set(feature_columns + target_columns + ["country", "station", "date"]) - set(df.columns))
assert not missing, f"Colunas ausentes: {missing}"
assert set(required_features).issubset(feature_columns)
assert feature_columns == model_manifest["features"]
assert target_columns == model_manifest["targets"]

coverage = df.groupby("country")[feature_columns].agg(lambda values: values.notna().sum())
display(coverage)

MIN_SAMPLES_PER_COUNTRY = 30
counts = df.groupby("country").size().rename("n_amostras")
eligible = counts[counts >= MIN_SAMPLES_PER_COUNTRY].index
profiles = df[df["country"].isin(eligible)].groupby("country")[feature_columns].median()
profiles = profiles.dropna(subset=required_features)
predictions = model.predict(profiles[feature_columns])
scenario_results = pd.DataFrame(predictions, index=profiles.index, columns=target_columns)
scenario_results.insert(0, "n_amostras", counts.loc[scenario_results.index])
print(f"Países incluídos: {len(scenario_results)} de {df['country'].nunique()} (corte: {MIN_SAMPLES_PER_COUNTRY} amostras)")
display(profiles)
display(scenario_results.sort_values("n_amostras", ascending=False).round(5))


In [ ]:
# Painéis separados preservam a unidade e a escala próprias de cada metal.
fig, axes = plt.subplots(4, 2, figsize=(14, 15), constrained_layout=True)
for ax, metal in zip(axes.flat, target_columns):
    values = scenario_results[metal].sort_values()
    ax.barh(values.index, values.values, color="#287c8e")
    ax.set_title(metal)
    ax.set_xlabel("Concentração estimada (unidade no título)")
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Estimativas para perfis químicos medianos por país", fontsize=15)
plt.show()


## Interpretação, limitações e reprodução

As barras mostram a resposta do modelo aos perfis medianos, não concentrações médias de metais observadas. Países não são features e as diferenças não isolam um efeito geográfico. O modelo estima a amostra atual, não uma data futura. O conjunto documentado não tem estações brasileiras; o R² é baixo para Ni, Cu, Zn e Pb. Rótulos abaixo do limite de detecção foram aproximados por LOD/2, e a transformação do alvo limita previsões à faixa observada no ajuste. Não conclua tendências futuras, surtos ou segurança da água a partir desta demonstração.

**Para reproduzir:** use Python 3.13 e instale `back-end/ia/app/requirements.txt`; abra `back-end/ia/notes/modelo_metais_waterpath.ipynb` e execute em ordem, a partir da raiz do repositório ou desta pasta. O notebook usa os dados locais em `data/`, confere os hashes e carrega o artefato versionado; não baixa dados nem retreina o modelo.

**Fontes:** [GEMStat GFQA v3 / Zenodo](https://doi.org/10.5281/zenodo.18459694), [manifesto dos dados](data/dataset_manifest.json), [serving](../app/services/metals.py), [treinamento](../training/notebooks/regression/MultiOutputRegressor_v2.ipynb), [métricas e limitações](../training/experiment_outputs/regression/metals_v2/README.md). Próximo passo recomendado: avaliação temporal e geográfica independente, incluindo estações brasileiras com entradas e referências medidas pareadas.
